In [ ]:
from huggingface_hub import DatasetCard


card = DatasetCard.load(root_2)

card.data['configs']

In [ ]:
import pandas as pd
from datasets import load_dataset
from utils.dataset_loader import mimic_loader
from utils.clean_cohort_df import simplify_race_column
from datetime import timedelta
import numpy as np
import gc
pd.set_option('display.max_column', 136)

root = "ADS599-Capstone/raw_data"
root_2 = "ADS599-Capstone/modeling_data"

df_patient = pd.read_parquet("data/empty_patient_state_reduced.parquet")
df_cohort = pd.read_parquet("data/cohort_df_with_triage_label_corrected.parquet")

In [ ]:
transfers = pd.read_parquet("data/transfers.parquet")

In [5]:
df_cohort = simplify_race_column(df_cohort)

In [9]:
df_cohort.columns

Index(['subject_id', 'ed_stay_id', 'hadm_id', 'ed_intime', 'ed_outtime',
       'disposition', 'race', 'arrival_transport', 'first_careunit',
       'first_icu_intime', 'cohort_label', 'gender', 'anchor_age',
       'anchor_year', 'age_at_visit', 'dod', 'admittime', 'dischtime',
       'admission_type', 'discharge_location', 'insurance', 'language',
       'marital_status', 'ed_stay_id_2', 'stay_window_start',
       'stay_window_end', 'ed_boarding_time_hrs', 'temperature', 'heartrate',
       'resprate', 'o2sat', 'sbp', 'dbp', 'pain', 'acuity', 'chiefcomplaint'],
      dtype='str')

time variable to track actions.  Take full `stay_window_start` and just add 10 min for each step  

Static Variables  
- race [x]  
- gender [x]  
- age [x]  
- arrival transport [x]  
- height [x]  
- weight [x]
- medications (0: Not on, 1: On) [x]  
- triage stats (except chiefcomplaint) [x]  
- admission type (if/when admitted to ward) [x]  

Dynamic Variables  

- Labs [x]  
    - 19 columns (one for each combo of category/fluid)
    - Code will be 0: Not Ordered, 1: Pending, 2: Normal, 3: Abnormal
    - Refresh with each new lab for that cat ordered

- Microbiology [x]  
    - 21 columns for each of the cultures to order
    - Code will be 0: Not Ordered, 1: Pending, 2: Negative, 3: Postive, 4: Other/Cancelled
    - Code and refresh is same as for labs

- Vitals [x]
    - Each vital will have the following
        - Current result
        - Change from previous
        - Rate of change
        - Running avg (2h)
        - blood pressure is adding MAP which is perfusion pressure and pulse pressure

- Observe [x]  
    - when not ordering anything else
    - Coded for 0: Not, 1: Observe

- ECG [x]  
    - One column in state, 0: Not ordered, 1: Normal, 2: Moderate, 3: Acute

- Radiology [x]  
    - Same as ECG

- Medications [x]
    - 22 columns for each major category of medications in data
    - Coding is 0: Not administered 1: Administered

- Transfer [x]
    - 3 actions, transfer_icu, transfer_ward or discharge
    - One-hot encode location of patient (ed, ward)
    - ICU and discharge are termination states -> For ICU transfer, need to still capture the end result (died, home etc)
    - Ward is represented in patient state.  Coded 0: ER 1: Ward
    - Carve out boarded patients (sitting in ICU, not transferred to ICU)

Actions filled in when the time occurs within the range between 2 time steps

Discrepancies between med actions and medrecon
{'analgesic_acetaminophen', 'iv_fluid', 'other'}

# Add Time for each block

In [10]:
# Merges the stay window start and end, adds a column with a cumulative time_block count for each grouping, adds that to stay window start time
# to track the actual time then replaces the last index with the patients actual end of stay time

time_block = 30

stay_times = df_cohort[['ed_stay_id', 'stay_window_start', 'stay_window_end']]
df_patient = df_patient.merge(stay_times, how='left', on='ed_stay_id')
df_patient['time'] = df_patient.groupby('ed_stay_id').cumcount() * timedelta(minutes=time_block)
df_patient['time'] = pd.to_datetime(df_patient['stay_window_start']) + df_patient['time']

last_idx = df_patient.groupby('ed_stay_id').tail(1).index
df_patient.loc[last_idx, 'time'] = df_patient.loc[last_idx, 'stay_window_end']

df_patient.drop(columns=['stay_window_start', 'stay_window_end', 'time_steps'], errors='ignore', inplace=True)

df_patient.head()

,ed_stay_id,subject_id,hadm_id,time
0,32952584,10000032,29079034.0,2180-07-22 16:24:00
1,32952584,10000032,29079034.0,2180-07-22 16:54:00
2,32952584,10000032,29079034.0,2180-07-22 17:24:00
3,32952584,10000032,29079034.0,2180-07-22 17:54:00
4,32952584,10000032,29079034.0,2180-07-22 18:24:00


# Adding Labs

In [12]:
df_labs = pd.read_csv("data/collapsed_labs.csv")

In [13]:
# create actions and add them as columns to dataframe
df_labs['action'] = df_labs['category'] + "-" + df_labs['fluid']
actions = df_labs['action'].value_counts().index.to_list()
df_patient[actions] = np.nan

# Change times to datetimes
df_labs['order_time'] = pd.to_datetime(df_labs['order_time'])
df_labs['result_time'] = pd.to_datetime(df_labs['result_time'])

df_patient.head()

,ed_stay_id,subject_id,hadm_id,time,Chemistry-Blood,Hematology-Blood,Hematology-Urine,Blood Gas-Blood,Chemistry-Urine,Chemistry-Other Body Fluid,Hematology-Cerebrospinal Fluid,Chemistry-Cerebrospinal Fluid,Hematology-Ascites,Chemistry-Ascites,Chemistry-Pleural,Hematology-Pleural,Hematology-Joint Fluid,Chemistry-Stool,Hematology-Other Body Fluid,Blood Gas-Other Body Fluid,Hematology-Bone Marrow,Hematology-Stool,Chemistry-Joint Fluid
0,32952584,10000032,29079034.0,2180-07-22 16:24:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,32952584,10000032,29079034.0,2180-07-22 16:54:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,32952584,10000032,29079034.0,2180-07-22 17:24:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,32952584,10000032,29079034.0,2180-07-22 17:54:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,32952584,10000032,29079034.0,2180-07-22 18:24:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df_patient.reset_index(drop=True, inplace=True)
df_patient['time'] = pd.to_datetime(df_patient['time'])
df_labs['order_time'] = pd.to_datetime(df_labs['order_time'])
df_labs['result_time'] = pd.to_datetime(df_labs['result_time'])
df_labs['action'] = df_labs['category'] + "-" + df_labs['fluid']

# Drop rows where merge keys are null
df_labs_orders = df_labs.dropna(subset=['order_time'])
df_labs_results = df_labs.dropna(subset=['result_time'])

df_patient_sorted = df_patient[['ed_stay_id', 'time']].reset_index().rename(columns={'index': 'step_idx'})

# Last index per stay to cap result fills
stay_last_idx = df_patient.groupby('ed_stay_id').tail(1).reset_index()[['ed_stay_id', 'index']].rename(columns={'index': 'last_idx'})

# Find time step where order was placed
order_steps = pd.merge_asof(
    df_labs_orders.sort_values('order_time'),
    df_patient_sorted.sort_values('time'),
    left_on='order_time',
    right_on='time',
    by='ed_stay_id',
    direction='backward'
).rename(columns={'step_idx': 'order_step'})

# Find time step where result came back
result_steps = pd.merge_asof(
    df_labs_results.sort_values('result_time'),
    df_patient_sorted.sort_values('time'),
    left_on='result_time',
    right_on='time',
    by='ed_stay_id',
    direction='backward'
).rename(columns={'step_idx': 'result_step'})

# Join result_step back onto order_steps
order_steps = order_steps.merge(
    result_steps[['ed_stay_id', 'order_time', 'action', 'result_step']],
    on=['ed_stay_id', 'order_time', 'action'],
    how='left'
)

order_steps['result_code'] = np.where(order_steps['abnormal'], 3, 2)
order_steps = order_steps.merge(stay_last_idx, on='ed_stay_id', how='left')
order_steps = order_steps.dropna(subset=['order_step', 'result_step', 'last_idx'])
order_steps[['order_step', 'result_step', 'last_idx']] = order_steps[['order_step', 'result_step', 'last_idx']].astype(int)

# Sort chronologically so most recent result wins when we take max
order_steps = order_steps.sort_values('order_time').reset_index(drop=True)

# labs_ordered: 1 at any step where a lab was ordered
labs_ordered_idx = set(order_steps['order_step'].dropna().astype(int))
df_patient['labs_ordered'] = df_patient.index.isin(labs_ordered_idx).astype(int)

# --- Fully vectorized expansion using cumsum offsets ---

# Pending windows (order_step to result_step - 1)
pending = order_steps[order_steps['result_step'] > order_steps['order_step']].copy()
pending_lengths = pending['result_step'] - pending['order_step']
pending_offsets = np.arange(pending_lengths.sum())
pending_offsets -= np.repeat(pending_lengths.cumsum().shift(1, fill_value=0).values, pending_lengths.values)
pending_expanded = pending.loc[pending.index.repeat(pending_lengths)].copy()
pending_expanded['step_idx'] = np.repeat(pending['order_step'].values, pending_lengths.values) + pending_offsets
pending_expanded['status'] = 1

# Result windows (result_step to last_idx)
result = order_steps[order_steps['last_idx'] >= order_steps['result_step']].copy()
result_lengths = result['last_idx'] - result['result_step'] + 1
result_offsets = np.arange(result_lengths.sum())
result_offsets -= np.repeat(result_lengths.cumsum().shift(1, fill_value=0).values, result_lengths.values)
result_expanded = result.loc[result.index.repeat(result_lengths)].copy()
result_expanded['step_idx'] = np.repeat(result['result_step'].values, result_lengths.values) + result_offsets
result_expanded['status'] = result_expanded['result_code']

# Combine: max status per (step_idx, action) 2/3 beats 1
all_expanded = pd.concat([pending_expanded, result_expanded])[['step_idx', 'action', 'status']]
all_expanded = all_expanded.groupby(['step_idx', 'action'])['status'].max().reset_index()

# Pivot to wide format
status_pivot = all_expanded.pivot(index='step_idx', columns='action', values='status')

# Assign back to df_patient
for col in status_pivot.columns:
    if col in df_patient.columns:
        df_patient.loc[status_pivot.index, col] = status_pivot[col].values

# Fill remaining NaN with 0 (not ordered)
df_patient[actions] = df_patient[actions].fillna(0)

## Checks

In [15]:
# 1. No action column should have values outside {0, 1, 2, 3}
assert df_patient[actions].isin([0, 1, 2, 3]).all().all()

# 2. No NaNs remaining in action columns
assert df_patient[actions].isna().sum().sum() == 0

# 3. Within each stay, no action should go from 2/3 back to 1
# (result shouldn't revert to pending)
def check_no_revert(group):
    for col in actions:
        vals = group[col].values
        for j in range(1, len(vals)):
            if vals[j] == 1 and vals[j-1] in (2, 3):
                return False
    return True

# spot check on a few stays
sample_ids = df_patient['ed_stay_id'].unique()[:20]
assert all(check_no_revert(g) for _, g in df_patient[df_patient['ed_stay_id'].isin(sample_ids)].groupby('ed_stay_id'))

# 4. Quick value distribution
df_patient[actions].apply(lambda x: x.value_counts()).T


,0.0,1.0,2.0,3.0
Chemistry-Blood,500628.0,399270.0,411340.0,12686274.0
Hematology-Blood,509291.0,120737.0,179054.0,13188430.0
Hematology-Urine,4847295.0,65813.0,2338561.0,6745843.0
Blood Gas-Blood,4085274.0,12247.0,3432459.0,6467532.0
Chemistry-Urine,10353025.0,136444.0,2985830.0,522213.0
Chemistry-Other Body Fluid,12880103.0,36955.0,1047484.0,32970.0
Hematology-Cerebrospinal Fluid,13743853.0,5062.0,56405.0,192192.0
Chemistry-Cerebrospinal Fluid,13746450.0,4856.0,105472.0,140734.0
Hematology-Ascites,13526358.0,5607.0,5132.0,460415.0
Chemistry-Ascites,13562876.0,5885.0,428751.0,NaN


In [ ]:
del df_labs
gc.collect()

969

# Microbiology Events

In [23]:
df_micro = pd.read_csv('data/microbiology_events_collapsed.csv')

In [24]:
df_micro.head()

,subject_id,ed_stay_id,hadm_id,charttime,storetime,action_space,culture_result
0,11714491,30000012,21562392.0,2126-02-14 22:58:00,2126-02-16 09:06:00,URINE,POSITIVE
1,13821532,30000038,26255538.0,2152-12-08 04:51:00,2152-12-09 11:46:00,Rapid Respiratory Viral Screen & Culture,OTHER
2,13821532,30000038,26255538.0,2152-12-08 14:30:00,2152-12-09 04:00:00,URINE,NEGATIVE
3,13658097,30000317,23069398.0,2138-06-09 19:55:00,2138-06-11 11:12:00,URINE,POSITIVE
4,15293245,30000379,21532833.0,2136-07-20 03:22:00,2136-07-21 10:43:00,URINE,POSITIVE


In [25]:
# create actions and add them as columns to dataframe
actions = list(df_micro['action_space'].unique())
df_patient[actions] = np.nan

# Change times to datetimes
df_micro['charttime'] = pd.to_datetime(df_micro['charttime'])
df_micro['storetime'] = pd.to_datetime(df_micro['storetime'])

df_patient.head()

,ed_stay_id,subject_id,hadm_id,time,Chemistry-Blood,Hematology-Blood,Hematology-Urine,Blood Gas-Blood,Chemistry-Urine,Chemistry-Other Body Fluid,Hematology-Cerebrospinal Fluid,Chemistry-Cerebrospinal Fluid,Hematology-Ascites,Chemistry-Ascites,Chemistry-Pleural,Hematology-Pleural,Hematology-Joint Fluid,Chemistry-Stool,Hematology-Other Body Fluid,Blood Gas-Other Body Fluid,Hematology-Bone Marrow,Hematology-Stool,Chemistry-Joint Fluid,labs_ordered,URINE,Rapid Respiratory Viral Screen & Culture,PERITONEAL FLUID,BLOOD CULTURE,STOOL,OTHER,PLEURAL FLUID,TISSUE,SEROLOGY/BLOOD,IMMUNOLOGY,MRSA SCREEN,SWAB,JOINT FLUID,SPUTUM,"FLUID,OTHER",ABSCESS,BRONCHOALVEOLAR LAVAGE,CSF;SPINAL FLUID,Blood (CMV AB),Blood (EBV),BILE
0,32952584,10000032,29079034.0,2180-07-22 16:24:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,32952584,10000032,29079034.0,2180-07-22 16:54:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,32952584,10000032,29079034.0,2180-07-22 17:24:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,32952584,10000032,29079034.0,2180-07-22 17:54:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,32952584,10000032,29079034.0,2180-07-22 18:24:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_patient.reset_index(drop=True, inplace=True)
micro_actions = list(df_micro['action_space'].unique())

# Result code mapping
result_map = {'NEGATIVE': 2, 'POSITIVE': 3, 'OTHER': 4, 'CANCELLED': 4}
df_micro['result_code'] = df_micro['culture_result'].map(result_map)

# Drop rows where merge keys are null
df_micro_orders = df_micro.dropna(subset=['charttime'])
df_micro_results = df_micro.dropna(subset=['storetime'])

df_patient_sorted = df_patient[['ed_stay_id', 'time']].reset_index().rename(columns={'index': 'step_idx'})

# Last index per stay to cap result fills
stay_last_idx = df_patient.groupby('ed_stay_id').tail(1).reset_index()[['ed_stay_id', 'index']].rename(columns={'index': 'last_idx'})

# Find time step where culture was ordered
order_steps = pd.merge_asof(
    df_micro_orders.sort_values('charttime'),
    df_patient_sorted.sort_values('time'),
    left_on='charttime',
    right_on='time',
    by='ed_stay_id',
    direction='backward'
).rename(columns={'step_idx': 'order_step'})

# Find time step where result came back (drop result_code to avoid duplicate on merge)
result_steps = pd.merge_asof(
    df_micro_results.sort_values('storetime'),
    df_patient_sorted.sort_values('time'),
    left_on='storetime',
    right_on='time',
    by='ed_stay_id',
    direction='backward'
).rename(columns={'step_idx': 'result_step'})

# Join result_step and result_code back onto order_steps using result_code from result_steps only
order_steps = order_steps.drop(columns=['result_code']).merge(
    result_steps[['ed_stay_id', 'charttime', 'action_space', 'result_step', 'result_code']],
    on=['ed_stay_id', 'charttime', 'action_space'],
    how='left'
)

order_steps = order_steps.merge(stay_last_idx, on='ed_stay_id', how='left')
order_steps = order_steps.dropna(subset=['order_step', 'result_step', 'last_idx', 'result_code'])
order_steps[['order_step', 'result_step', 'last_idx', 'result_code']] = order_steps[['order_step', 'result_step', 'last_idx', 'result_code']].astype(int)
order_steps = order_steps.sort_values('charttime').reset_index(drop=True)

# micro_ordered: 1 at any step where a micro culture was ordered
micro_ordered_idx = set(order_steps['order_step'].dropna().astype(int))
df_patient['micro_ordered'] = df_patient.index.isin(micro_ordered_idx).astype(int)

# --- Fully vectorized expansion ---

# Pending windows (order_step to result_step - 1)
pending = order_steps[order_steps['result_step'] > order_steps['order_step']].copy()
pending_lengths = pending['result_step'] - pending['order_step']
pending_offsets = np.arange(pending_lengths.sum())
pending_offsets -= np.repeat(pending_lengths.cumsum().shift(1, fill_value=0).values, pending_lengths.values)
pending_expanded = pending.loc[pending.index.repeat(pending_lengths)].copy()
pending_expanded['step_idx'] = np.repeat(pending['order_step'].values, pending_lengths.values) + pending_offsets
pending_expanded['status'] = 1

# Result windows (result_step to last_idx)
result = order_steps[order_steps['last_idx'] >= order_steps['result_step']].copy()
result_lengths = result['last_idx'] - result['result_step'] + 1
result_offsets = np.arange(result_lengths.sum())
result_offsets -= np.repeat(result_lengths.cumsum().shift(1, fill_value=0).values, result_lengths.values)
result_expanded = result.loc[result.index.repeat(result_lengths)].copy()
result_expanded['step_idx'] = np.repeat(result['result_step'].values, result_lengths.values) + result_offsets
result_expanded['status'] = result_expanded['result_code']

# Combine: max status per (step_idx, action_space) â€” result beats pending
all_expanded = pd.concat([pending_expanded, result_expanded])[['step_idx', 'action_space', 'status']]
all_expanded = all_expanded.groupby(['step_idx', 'action_space'])['status'].max().reset_index()

# Pivot to wide format
status_pivot = all_expanded.pivot(index='step_idx', columns='action_space', values='status')

# Assign back to df_patient
for col in status_pivot.columns:
    if col in df_patient.columns:
        df_patient.loc[status_pivot.index, col] = status_pivot[col].values

# Fill remaining NaN with 0 (not ordered)
df_patient[micro_actions] = df_patient[micro_actions].fillna(0)
del all_expanded

df_patient.head(10)

,ed_stay_id,subject_id,hadm_id,time,Chemistry-Blood,Hematology-Blood,Hematology-Urine,Blood Gas-Blood,Chemistry-Urine,Chemistry-Other Body Fluid,Hematology-Cerebrospinal Fluid,Chemistry-Cerebrospinal Fluid,Hematology-Ascites,Chemistry-Ascites,Chemistry-Pleural,Hematology-Pleural,Hematology-Joint Fluid,Chemistry-Stool,Hematology-Other Body Fluid,Blood Gas-Other Body Fluid,Hematology-Bone Marrow,Hematology-Stool,Chemistry-Joint Fluid,labs_ordered,URINE,Rapid Respiratory Viral Screen & Culture,PERITONEAL FLUID,BLOOD CULTURE,STOOL,OTHER,PLEURAL FLUID,TISSUE,SEROLOGY/BLOOD,IMMUNOLOGY,MRSA SCREEN,SWAB,JOINT FLUID,SPUTUM,"FLUID,OTHER",ABSCESS,BRONCHOALVEOLAR LAVAGE,CSF;SPINAL FLUID,Blood (CMV AB),Blood (EBV),BILE,micro_ordered
0,32952584,10000032,29079034.0,2180-07-22 16:24:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,32952584,10000032,29079034.0,2180-07-22 16:54:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,32952584,10000032,29079034.0,2180-07-22 17:24:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,32952584,10000032,29079034.0,2180-07-22 17:54:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,32952584,10000032,29079034.0,2180-07-22 18:24:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
5,32952584,10000032,29079034.0,2180-07-22 18:54:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
6,32952584,10000032,29079034.0,2180-07-22 19:24:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
7,32952584,10000032,29079034.0,2180-07-22 19:54:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
8,32952584,10000032,29079034.0,2180-07-22 20:24:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
9,32952584,10000032,29079034.0,2180-07-22 20:54:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


## Checks

In [27]:
# 1. No action column should have values outside {0, 1, 2, 3, 4}
assert df_patient[micro_actions].isin([0, 1, 2, 3, 4]).all().all(), "Unexpected values in micro action columns"

# 2. No NaNs remaining
assert df_patient[micro_actions].isna().sum().sum() == 0, "NaNs found in micro action columns"

# 3. No revert from result back to pending within a stay (spot check)
def check_no_revert(group):
    for col in micro_actions:
        vals = group[col].values
        for j in range(1, len(vals)):
            if vals[j] == 1 and vals[j-1] in (2, 3, 4):
                return False
    return True

sample_ids = df_patient['ed_stay_id'].unique()[:20]
assert all(
    check_no_revert(g)
    for _, g in df_patient[df_patient['ed_stay_id'].isin(sample_ids)].groupby('ed_stay_id')
), "Result reverted to pending in at least one stay"

# 4. Value distribution across all micro action columns
df_patient[micro_actions].apply(lambda x: x.value_counts()).T

,0.0,1.0,2.0,3.0,4.0
URINE,10009617.0,728238.0,1114109.0,2135408.0,10140.0
Rapid Respiratory Viral Screen & Culture,13820796.0,35998.0,85120.0,10155.0,45443.0
PERITONEAL FLUID,13578138.0,98081.0,286459.0,19644.0,15190.0
BLOOD CULTURE,11355500.0,1004969.0,1241082.0,244621.0,151340.0
STOOL,13107313.0,124780.0,564592.0,90682.0,110145.0
OTHER,13383021.0,122158.0,357222.0,75017.0,60094.0
PLEURAL FLUID,13820742.0,56170.0,105693.0,10856.0,4051.0
TISSUE,13734893.0,97127.0,81991.0,67930.0,15571.0
SEROLOGY/BLOOD,13601928.0,53227.0,278171.0,49693.0,14493.0
IMMUNOLOGY,13854321.0,23763.0,79425.0,5415.0,34588.0


In [28]:
del df_micro
gc.collect()

16

# Vitals

In [ ]:
df_vitals = mimic_loader(path=root_2, name='vitals')
df_vitals.drop_duplicates(subset=['ed_stay_id', 'charttime'], inplace=True)

In [ ]:
df_vitals.head()

In [124]:
df_vitals['charttime'] = pd.to_datetime(df_vitals['charttime'])

# Round rate columns to 2 decimal places
rate_cols = [c for c in df_vitals.columns[10:]]
df_vitals[rate_cols] = df_vitals[rate_cols].round(2)

# Rename singular vital columns to current_*
rename_map = {
    'temperature': 'current_temperature',
    'heartrate': 'current_heartrate',
    'resprate': 'current_resprate',
    'o2sat': 'current_o2sat',
    'sbp': 'current_sbp',
    'dbp': 'current_dbp',
    'pain': 'current_pain',
    'pulse_pressure': 'current_pulse_pressure',
    'map': 'current_map',
}
df_vitals = df_vitals.rename(columns=rename_map)

vital_cols = [
    'current_temperature', 'current_heartrate', 'current_resprate',
    'current_o2sat', 'current_sbp', 'current_dbp', 'current_pain',
    'current_pulse_pressure', 'current_map', 'time_since_last_hrs',
    'temperature_rolling2h', 'temperature_delta', 'temperature_rate',
    'heartrate_rolling2h', 'heartrate_delta', 'heartrate_rate',
    'resprate_rolling2h', 'resprate_delta', 'resprate_rate',
    'o2sat_rolling2h', 'o2sat_delta', 'o2sat_rate',
    'sbp_rolling2h', 'sbp_delta', 'sbp_rate',
    'dbp_rolling2h', 'dbp_delta', 'dbp_rate',
]

df_patient_sorted = df_patient[['ed_stay_id', 'time']].reset_index().rename(columns={'index': 'step_idx'})

# Snap each vitals reading to its nearest preceding time step
vitals_snapped = pd.merge_asof(
    df_vitals.sort_values('charttime'),
    df_patient_sorted.sort_values('time'),
    left_on='charttime',
    right_on='time',
    by='ed_stay_id',
    direction='backward'
).rename(columns={'time': 'step_time'})

# If multiple vitals fall in the same time step, keep the last reading
vitals_snapped = (
    vitals_snapped.sort_values('charttime')
    .groupby(['ed_stay_id', 'step_idx'], as_index=False)
    .last()
    .set_index('step_idx')
)

# Capture which step indices had actual measurements before any ffill
measured_idx = set(vitals_snapped.index)

# Merge vital values onto df_patient by index
df_patient = df_patient.merge(
    vitals_snapped[vital_cols],
    left_index=True,
    right_index=True,
    how='left'
)

# Fill forward raw/rolling vitals within each stay
ffill_cols = [
    'current_temperature', 'current_heartrate', 'current_resprate',
    'current_o2sat', 'current_sbp', 'current_dbp', 'current_pain',
    'current_pulse_pressure', 'current_map',
    'temperature_rolling2h', 'heartrate_rolling2h', 'resprate_rolling2h',
    'o2sat_rolling2h', 'sbp_rolling2h', 'dbp_rolling2h',
]
df_patient[ffill_cols] = df_patient.groupby('ed_stay_id')[ffill_cols].ffill()

# delta and rate cols: fill forward only (NaN at start of stay preserved as structural missing)
delta_rate_cols = [c for c in vital_cols if c.endswith('_delta') or c.endswith('_rate')]
df_patient[delta_rate_cols] = df_patient.groupby('ed_stay_id')[delta_rate_cols].ffill()

# time_since_last_hrs: increment 0.5 hrs per step, use actual value at measurement time steps
df_patient['time_since_last_hrs'] = df_patient.groupby('ed_stay_id').cumcount() * 0.5
measured_mask = df_patient.index.isin(measured_idx)
df_patient.loc[measured_mask, 'time_since_last_hrs'] = vitals_snapped['time_since_last_hrs'].reindex(
    df_patient.loc[measured_mask].index
).values

# vitals_checked = 1 only at actual measurement steps, set last so ffill doesn't affect it
df_patient.insert(
    df_patient.columns.get_loc('current_temperature'),
    'vitals_checked',
    df_patient.index.isin(measured_idx).astype(float)
)

df_patient.head(10)

,ed_stay_id,subject_id,hadm_id,time,Chemistry-Blood,Hematology-Blood,Hematology-Urine,Blood Gas-Blood,Chemistry-Urine,Chemistry-Other Body Fluid,Hematology-Cerebrospinal Fluid,Chemistry-Cerebrospinal Fluid,Hematology-Ascites,Chemistry-Ascites,Chemistry-Pleural,Hematology-Pleural,Hematology-Joint Fluid,Chemistry-Stool,Hematology-Other Body Fluid,Blood Gas-Other Body Fluid,Hematology-Bone Marrow,Hematology-Stool,Chemistry-Joint Fluid,labs_ordered,URINE,Rapid Respiratory Viral Screen & Culture,PERITONEAL FLUID,BLOOD CULTURE,STOOL,OTHER,PLEURAL FLUID,TISSUE,SEROLOGY/BLOOD,IMMUNOLOGY,MRSA SCREEN,SWAB,JOINT FLUID,SPUTUM,"FLUID,OTHER",ABSCESS,BRONCHOALVEOLAR LAVAGE,CSF;SPINAL FLUID,Blood (CMV AB),Blood (EBV),BILE,micro_ordered,Other,Analgesic - Opioid/NSAID,Antiemetic,Analgesic - Acetaminophen,Antibiotic,Benzodiazepine - Sedative/Anxiolytic,Analgesic - NSAID,Bronchodilator,Antiplatelet,GI - Acid Suppression,Corticosteroid,Beta Blocker,Diuretic,Antipsychotic,Anticoagulant,Insulin/Glucose,Calcium Channel Blocker,Nitrate,ACE Inhibitor,IV Fluid,Anticonvulsant,Antiarrhythmic,weight,ecg_status,rad_status,in_ed,in_ward,ed_boarding,terminal_event,gender,anchor_age,arrival_transport,acuity,height,recon_ace_inhibitor,recon_analgesic___nsaid,recon_analgesic___opioid_nsaid,recon_antiarrhythmic,recon_antibiotic,recon_anticoagulant,recon_anticonvulsant,recon_antiemetic,recon_antiplatelet,recon_antipsychotic,recon_benzodiazepine___sedative_anxiolytic,recon_beta_blocker,recon_bronchodilator,recon_calcium_channel_blocker,recon_corticosteroid,recon_diuretic,recon_gi___acid_suppression,recon_insulin_glucose,recon_nitrate,admission_type,vitals_checked,current_temperature,current_heartrate,current_resprate,current_o2sat,current_sbp,current_dbp,current_pain,current_pulse_pressure,current_map,time_since_last_hrs,temperature_rolling2h,temperature_delta,temperature_rate,heartrate_rolling2h,heartrate_delta,heartrate_rate,resprate_rolling2h,resprate_delta,resprate_rate,o2sat_rolling2h,o2sat_delta,o2sat_rate,sbp_rolling2h,sbp_delta,sbp_rate,dbp_rolling2h,dbp_delta,dbp_rate
0.0,30000012,11714491,21562392.0,2126-02-14 20:22:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,1,0,0,DISCHARGE_WARD,F,65,AMBULANCE,2.000000000,65.0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,1,0,0,NaN,1.0,98.8,96.0,18.0,93.0,160.0,54.0,0.0,106.0,89.33,0.00,98.8,0.0,NaN,96.0,0.0,NaN,18.0,0.0,NaN,93.0,0.0,NaN,160.0,0.0,NaN,54.0,0.0,NaN
1.0,30000012,11714491,21562392.0,2126-02-14 20:52:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,3.0,0.0,1,0,0,DISCHARGE_WARD,F,65,AMBULANCE,2.000000000,65.0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,1,0,0,NaN,0.0,98.8,96.0,18.0,93.0,160.0,54.0,0.0,106.0,89.33,0.50,98.8,0.0,NaN,96.0,0.0,NaN,18.0,0.0,NaN,93.0,0.0,NaN,160.0,0.0,NaN,54.0,0.0,NaN
2.0,30000012,11714491,21562392.0,2126-02-14 21:22:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,3.0,0.0,1,0,0,DISCHARGE_WARD,F,65,AMBULANCE,2.000000000,65.0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,1,0,0,NaN,0.0,98.8,96.0,18.0,93.0,160.0,54.0,0.0,106.0,89.33,1.00,98.8,0.0,NaN,96.0,0.0,NaN,18.0,0.0,NaN,93.0,0.0,NaN,160.0,0.0,NaN,54.0,0.0,NaN
3.0,30000012,11714491,21562392.0,2126-02-14 21:52:00,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0

## Checks

In [ ]:
# 1. vitals_checked should only be 0 or 1
assert df_patient['vitals_checked'].isin([0, 1]).all(), "Unexpected values in vitals_checked"

# 2. vitals_checked count should match number of unique (ed_stay_id, step_idx) in vitals_snapped
assert df_patient['vitals_checked'].sum() == len(measured_idx), \
    f"vitals_checked count {df_patient['vitals_checked'].sum()} != measured steps {len(measured_idx)}"

# 3. For stays longer than their vitals count, vitals_checked should not be 1 for every step
check = df_patient.groupby('ed_stay_id')['vitals_checked'].agg(['sum', 'count'])
check['pct_checked'] = check['sum'] / check['count']
long_stays = check[check['count'] > check['sum']]
assert (long_stays['pct_checked'] < 1.0).all(), "Some long stays have vitals_checked=1 for every time step"

# 4. ffill cols should have no NaN after the first measurement in each stay
def check_no_nan_after_first(group):
    for col in ffill_cols:
        first_valid = group[col].first_valid_index()
        if first_valid is not None:
            if group.loc[first_valid:, col].isna().any():
                return False
    return True

sample_ids = df_patient['ed_stay_id'].unique()[:20]
assert all(
    check_no_nan_after_first(g)
    for _, g in df_patient[df_patient['ed_stay_id'].isin(sample_ids)].groupby('ed_stay_id')
), "NaN found after first valid reading in ffill columns"

# 5. time_since_last_hrs should start at 0, reset (drop) after each vitals check
def check_time_since(group):
    if group['time_since_last_hrs'].iloc[0] != 0.0:
        return False
    # After each vitals check, the next step should be <= 0.5 (reset + one increment)
    checked_idx = group.index[group['vitals_checked'] == 1]
    for idx in checked_idx:
        loc = group.index.get_loc(idx)
        if loc + 1 < len(group):
            next_val = group['time_since_last_hrs'].iloc[loc + 1]
            if next_val > 1.0:  # allow small tolerance for actual measured value at check time
                return False
    return True

# it's currently set up so that when vitals are checked again, the time is still the time since last check. So it doesn't reset,
# just decreases in the time step after vitals checked

# assert all(
#     check_time_since(g)
#     for _, g in df_patient[df_patient['ed_stay_id'].isin(sample_ids)].groupby('ed_stay_id')
# ), "time_since_last_hrs does not reset after vitals check"

# 6. Summary
print("vitals_checked distribution:")
print(df_patient['vitals_checked'].value_counts())
print("\nNaN counts per vital column:")
print(df_patient[vital_cols].isna().sum())
print("\nMeasurements per stay (sample):")
print(check[['sum', 'count', 'pct_checked']].head(10))

In [37]:
del df_vitals
gc.collect()

545

# Medication Actions

In [38]:
df_med_actions = load_dataset(path=root_2, name='all_med_actions', split='all_med_actions').to_pandas()

In [ ]:
df_med_actions.head()

In [39]:
print(df_med_actions.duplicated().sum())
df_med_actions.drop_duplicates(inplace=True)

320827


In [40]:
medication_actions = df_med_actions.value_counts('drug_class').index.to_list()
df_patient[medication_actions] = np.nan

df_med_actions['charttime'] = pd.to_datetime(df_med_actions['charttime'])

In [42]:
df_patient_sorted = df_patient[['ed_stay_id', 'time']].reset_index().rename(columns={'index': 'step_idx'})

# Align data types before merge
df_med_actions['ed_stay_id'] = df_med_actions['ed_stay_id'].astype(int)

df_med_actions['charttime'] = df_med_actions['charttime'].astype('datetime64[us]')
df_patient_sorted['time'] = df_patient_sorted['time'].astype('datetime64[us]')

# Snap each medication administration to its nearest preceding time step
med_snapped = pd.merge_asof(
    df_med_actions.sort_values('charttime'),
    df_patient_sorted.sort_values('time'),
    left_on='charttime',
    right_on='time',
    by='ed_stay_id',
    direction='backward'
).dropna(subset=['step_idx'])

med_snapped['step_idx'] = med_snapped['step_idx'].astype(int)

# Last index per stay to cap forward fill
stay_last_idx = df_patient_sorted.groupby('ed_stay_id')['step_idx'].max().reset_index().rename(columns={'step_idx': 'last_idx'})
med_snapped = med_snapped.merge(stay_last_idx, on='ed_stay_id', how='left')

# Keep only the first administration per (ed_stay_id, drug_class) â€” once given, stays 1
first_admin = (
    med_snapped.sort_values('charttime')
    .groupby(['ed_stay_id', 'drug_class'], as_index=False)
    .first()
)
first_admin[['step_idx', 'last_idx']] = first_admin[['step_idx', 'last_idx']].astype(int)

# Vectorized expansion: from first admin step to end of stay
admin_lengths = (first_admin['last_idx'] - first_admin['step_idx'] + 1).clip(lower=0)
admin_expanded = first_admin.loc[first_admin.index.repeat(admin_lengths)].copy()
admin_offsets = np.arange(admin_lengths.sum())
admin_offsets -= np.repeat(admin_lengths.cumsum().shift(1, fill_value=0).values, admin_lengths.values)
admin_expanded['row_idx'] = np.repeat(first_admin['step_idx'].values, admin_lengths.values) + admin_offsets
admin_expanded['status'] = 1

# Pivot and assign back
status_pivot = admin_expanded.pivot_table(index='row_idx', columns='drug_class', values='status', aggfunc='max')

for col in status_pivot.columns:
    if col in df_patient.columns:
        df_patient.loc[status_pivot.index, col] = status_pivot[col].values

# Fill remaining NaN with 0 (not administered)
df_patient[medication_actions] = df_patient[medication_actions].fillna(0)
del admin_expanded

df_patient.head(10)

,ed_stay_id,subject_id,hadm_id,time,Chemistry-Blood,Hematology-Blood,Hematology-Urine,Blood Gas-Blood,Chemistry-Urine,Chemistry-Other Body Fluid,Hematology-Cerebrospinal Fluid,Chemistry-Cerebrospinal Fluid,Hematology-Ascites,Chemistry-Ascites,Chemistry-Pleural,Hematology-Pleural,Hematology-Joint Fluid,Chemistry-Stool,Hematology-Other Body Fluid,Blood Gas-Other Body Fluid,Hematology-Bone Marrow,Hematology-Stool,Chemistry-Joint Fluid,labs_ordered,URINE,Rapid Respiratory Viral Screen & Culture,PERITONEAL FLUID,BLOOD CULTURE,STOOL,OTHER,PLEURAL FLUID,TISSUE,SEROLOGY/BLOOD,IMMUNOLOGY,MRSA SCREEN,SWAB,JOINT FLUID,SPUTUM,"FLUID,OTHER",ABSCESS,BRONCHOALVEOLAR LAVAGE,CSF;SPINAL FLUID,Blood (CMV AB),Blood (EBV),BILE,micro_ordered,vitals_checked,current_temperature,current_heartrate,current_resprate,current_o2sat,current_sbp,current_dbp,current_pain,current_pulse_pressure,current_map,time_since_last_hrs,temperature_rolling2h,temperature_delta,temperature_rate,heartrate_rolling2h,heartrate_delta,heartrate_rate,resprate_rolling2h,resprate_delta,resprate_rate,o2sat_rolling2h,o2sat_delta,o2sat_rate,sbp_rolling2h,sbp_delta,sbp_rate,dbp_rolling2h,dbp_delta,dbp_rate,Other,Analgesic - Opioid/NSAID,Antiemetic,Analgesic - Acetaminophen,Antibiotic,Benzodiazepine - Sedative/Anxiolytic,Analgesic - NSAID,Bronchodilator,Antiplatelet,GI - Acid Suppression,Corticosteroid,Beta Blocker,Diuretic,Antipsychotic,Anticoagulant,Insulin/Glucose,Calcium Channel Blocker,Nitrate,ACE Inhibitor,IV Fluid,Anticonvulsant,Antiarrhythmic
0.0,32952584,10000032,29079034.0,2180-07-22 16:24:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,98.4,84.0,22.0,97.0,75.0,39.0,0.0,36.0,51.00,0.03,98.25,0.0,0.00,84.75,-1.0,-30.0,20.5,0.0,0.00,97.25,-1.0,-30.00,78.00,-1.0,-30.00,43.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1.0,32952584,10000032,29079034.0,2180-07-22 16:54:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,98.4,84.0,22.0,97.0,75.0,39.0,0.0,36.0,51.00,0.50,98.25,0.0,0.00,84.75,-1.0,-30.0,20.5,0.0,0.00,97.25,-1.0,-30.00,78.00,-1.0,-30.00,43.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2.0,32952584,10000032,29079034.0,2180-07-22 17:24:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,98.4,84.0,22.0,97.0,75.0,39.0,0.0,36.0,51.00,1.00,98.25,0.0,0.00,84.75,-1.0,-30.0,20.5,0.0,0.00,97.25,-1.0,-30.00,78.00,-1.0,-30.00,43.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3.0,32952584,10000032,29079034.0,2180-07-22 17:54:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,98.4,84.0,20.0,99.0,86.0,51.0,Other,35.0,62.67,1.18,98.28,0.0,0.00,84.60,0.0,0.0,20.4,-2.0,-1.69,97.60,2.0,1.69,79.60,11.0,9.30,44.60,12.0,10.14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4.0,32952584,10000032,29079034.0,2180-07-22 18:24:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,98.4,85.0,16.0,99.0,83.0,45.0,0.0,38.0,57.67,0.02,98.40,0.0,0.00,84.80,-1.0,-60.0,20.0,-4.0,-240.00,98.20,1.0,60.00,77.00,18.0,1080.00,42.20,8.0,480.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5.0,32952584,10000032,29079034.0,2180-07-22 18:54:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0

## Checks

In [43]:
# 1. No values outside {0, 1}
assert df_patient[medication_actions].isin([0, 1]).all().all(), "Unexpected values in medication columns"

# 2. No NaNs
assert df_patient[medication_actions].isna().sum().sum() == 0, "NaNs found in medication columns"

# 3. Once a medication is 1, it should never go back to 0 within a stay
def check_no_revert(group):
    for col in medication_actions:
        vals = group[col].values
        turned_on = False
        for v in vals:
            if v == 1:
                turned_on = True
            if turned_on and v == 0:
                return False, col
    return True, None

sample_ids = df_patient['ed_stay_id'].unique()[:50]
for stay_id, group in df_patient[df_patient['ed_stay_id'].isin(sample_ids)].groupby('ed_stay_id'):
    ok, col = check_no_revert(group)
    assert ok, f"Medication {col} reverted to 0 after being 1 in stay {stay_id}"

# 4. Cross-check against df_med_actions: verify timing and drug class assignment
# Only sample stays present in both dataframes
shared_stays = df_med_actions[df_med_actions['ed_stay_id'].isin(df_patient['ed_stay_id'])]['ed_stay_id'].unique()[:5]

for stay_id in shared_stays:
    stay_meds = df_med_actions[df_med_actions['ed_stay_id'] == stay_id]
    stay_rows = df_patient[df_patient['ed_stay_id'] == stay_id].copy()

    print(f"\n--- Stay {stay_id} ---")
    for drug_class in stay_meds['drug_class'].unique():
        first_admin_time = stay_meds[stay_meds['drug_class'] == drug_class]['charttime'].min()
        first_coded_step = stay_rows[stay_rows[drug_class] == 1]['time'].min() if drug_class in stay_rows.columns else None
        print(f"  {drug_class}: first admin={first_admin_time}, first coded step={first_coded_step}")

# 5. Check that stays with NO medication for a class are coded as 0 throughout
stays_without_antibiotic = df_patient[
    ~df_patient['ed_stay_id'].isin(df_med_actions[df_med_actions['drug_class'] == 'Antibiotic']['ed_stay_id'])
]['ed_stay_id'].unique()[:10]

for stay_id in stays_without_antibiotic:
    stay_rows = df_patient[df_patient['ed_stay_id'] == stay_id]
    assert (stay_rows['Antibiotic'] == 0).all(), f"Stay {stay_id} coded as receiving Antibiotic but never administered"

print("\nAll checks passed.")


--- Stay 30000012 ---
  Other: first admin=2126-02-15 00:44:00, first coded step=2126-02-15 00:22:00
  Antibiotic: first admin=2126-02-15 01:22:00, first coded step=2126-02-15 01:22:00

--- Stay 30000112 ---
  Other: first admin=2157-12-12 15:02:00, first coded step=2157-12-12 14:45:00

--- Stay 30000177 ---
  Antiemetic: first admin=2143-12-28 00:03:00, first coded step=2143-12-27 23:50:00
  Other: first admin=2143-12-28 00:03:00, first coded step=2143-12-27 23:50:00
  Benzodiazepine - Sedative/Anxiolytic: first admin=2143-12-28 00:04:00, first coded step=2143-12-27 23:50:00

--- Stay 30000275 ---
  Antiemetic: first admin=2155-02-22 13:26:00, first coded step=2155-02-22 13:23:00
  Analgesic - Opioid/NSAID: first admin=2155-02-22 15:38:00, first coded step=2155-02-22 15:23:00
  Benzodiazepine - Sedative/Anxiolytic: first admin=2155-02-22 15:46:00, first coded step=2155-02-22 15:23:00

--- Stay 30000317 ---
  Analgesic - Opioid/NSAID: first admin=2138-06-09 19:46:00, first coded step=

In [44]:
del df_med_actions, admin_expanded
gc.collect()

505

# Weight

In [45]:
weight = mimic_loader(path=root, name='weight')
weight.head()

,subject_id,chartdate,result_name,result_value,__index_level_0__
0,12902507,2150-03-03,Weight,231.1,0
1,13215341,2193-06-18,Weight,178.0,1
2,14847279,2143-05-05,Weight,159.1,2
3,15439322,2134-08-02,Weight,174.6,3
4,10001725,2114-09-21,Weight,144.0,4


In [46]:
weight.drop(columns="__index_level_0__", inplace=True, errors='ignore')

# Floor chartdate to day to align with weight measurement granularity
weight['chartdate'] = pd.to_datetime(weight['chartdate']).dt.floor('D')

# Floor df_patient time to day for the merge key
df_patient_copy = df_patient[['subject_id', 'ed_stay_id', 'time']].copy()
df_patient_copy['new_time'] = pd.to_datetime(df_patient_copy['time']).dt.floor('D')

# Merge weight onto patient state by subject_id + day
new_weights = df_patient_copy.merge(
    weight,
    left_on=['subject_id', 'new_time'],
    right_on=['subject_id', 'chartdate'],
    how='left'
).rename(columns={'result_value': 'weight'}).drop(columns=['new_time', 'chartdate', 'result_name'])

# Fill forward within each subject â€” use only past measurements, no bfill (avoids future data leakage)
new_weights['weight'] = new_weights.groupby('subject_id')['weight'].transform('ffill')

# For subjects with no weight at all in the stay, use their mean weight across all available records
# (these are typically patients with weight measured outside the ED stay window)
all_null_subjects = new_weights['weight'].isna().groupby(new_weights['subject_id']).all()
ids_with_no_weights = all_null_subjects[all_null_subjects].index
mapping = weight[weight['subject_id'].isin(ids_with_no_weights)].groupby('subject_id')['result_value'].mean().round(2)
new_weights['weight'] = new_weights['weight'].fillna(new_weights['subject_id'].map(mapping))

# Note: remaining NaN weights are expected â€” ED-only patients discharged without admission have no weight records

# Assign weight column back to df_patient
df_patient = df_patient.merge(new_weights[['ed_stay_id', 'subject_id', 'time', 'weight']], on=['ed_stay_id', 'subject_id', 'time'], how='left')

print(f"Weight coverage: {df_patient['weight'].notna().mean():.1%} of time steps have a weight value")
print(f"Stays with no weight: {df_patient.groupby('ed_stay_id')['weight'].apply(lambda x: x.isna().all()).sum()}")

Weight coverage: 80.5% of time steps have a weight value
Stays with no weight: 23685


In [47]:
del weight

# ECG

In [48]:
df_ecg = load_dataset(path=root_2, name='ecg_features', split='ecg_features').to_pandas()

In [ ]:
df_ecg.head()

In [50]:
# Convert to time and drop the time zone so it's consistent
# Rename to ed_stay_id tp match patient state
# Add 1 to acuity to line up with coding standard for patient state

df_ecg['ecg_time'] = pd.to_datetime(df_ecg['ecg_time']).apply(lambda x: x.tz_localize(None))
df_ecg.rename(columns={'stay_id': "ed_stay_id"}, inplace=True)
df_ecg['ecg_acuity'] = df_ecg['ecg_acuity'] + 1

In [51]:
# Snap each ECG to its nearest preceding time step
df_patient = pd.merge_asof(
    df_patient.sort_values('time'),
    df_ecg.sort_values('ecg_time'),
    left_on='time',
    right_on='ecg_time',
    by=['ed_stay_id', 'subject_id', 'hadm_id'],
    direction='backward'
).rename(columns={'ecg_acuity': 'ecg_status'}).drop(columns='ecg_time')

# 0 = not ordered, fill remaining NaN with 0
df_patient['ecg_status'] = df_patient['ecg_status'].fillna(0)

In [52]:
del df_ecg

# Radiology

In [53]:
df_rad = load_dataset(path=root_2, name='radiology_features', split='radiology_features').to_pandas()
df_rad.head()

,subject_id,stay_id,hadm_id,exam_time,rad_acuity_level
0,11714491,30000012,21562392.0,2126-02-14 23:12:00+00:00,1
1,13340997,30000039,23100190.0,2165-10-06 13:40:00+00:00,2
2,19862552,30000094,NaN,2183-09-04 20:29:00+00:00,2
3,11615015,30000204,25540031.0,2132-10-10 07:58:00+00:00,2
4,19454512,30000262,NaN,2159-06-16 14:49:00+00:00,2


In [54]:
# Convert to time and drop the time zone so it's consistent
# Rename to ed_stay_id tp match patient state
# Add 1 to acuity to line up with coding standard for patient state

df_rad['exam_time'] = pd.to_datetime(df_rad['exam_time']).apply(lambda x: x.tz_localize(None))
df_rad.rename(columns={'stay_id': "ed_stay_id"}, inplace=True)
df_rad['rad_acuity_level'] = df_rad['rad_acuity_level'] + 1

In [55]:
# Snap each ECG to its nearest preceding time step
df_patient = pd.merge_asof(
    df_patient.sort_values('time'),
    df_rad.sort_values('exam_time'),
    left_on='time',
    right_on='exam_time',
    by=['ed_stay_id', 'subject_id', 'hadm_id'],
    direction='backward'
).rename(columns={'rad_acuity_level': 'rad_status'}).drop(columns='exam_time')

# 0 = not ordered, fill remaining NaN with 0
df_patient['rad_status'] = df_patient['rad_status'].fillna(0)

In [56]:
del df_rad

# Transfer/Discharge

In [58]:
patient_ids = list(df_patient['ed_stay_id'].unique())

In [59]:
cohort_subset = df_cohort[df_cohort['ed_stay_id'].isin(patient_ids)]
cohort_subset.head()

,subject_id,ed_stay_id,hadm_id,ed_intime,ed_outtime,disposition,race,arrival_transport,first_careunit,first_icu_intime,cohort_label,gender,anchor_age,anchor_year,age_at_visit,dod,admittime,dischtime,admission_type,discharge_location,insurance,language,marital_status,ed_stay_id_2,stay_window_start,stay_window_end,ed_boarding_time_hrs,temperature,heartrate,resprate,o2sat,sbp,dbp,pain,acuity,chiefcomplaint
0,10002013,34931809,21763296.0,2165-11-22 20:54:00,2165-11-23 08:20:42,ADMITTED,White,WALK IN,Medical Intensive Care Unit (MICU),2165-11-23 08:20:42,ED_DIRECT_ICU,F,53,2156,62,None,2165-11-23 08:19:00,2165-11-26 15:40:00,DIRECT EMER.,HOME HEALTH CARE,Medicaid,English,SINGLE,NaN,2165-11-22 20:54:00,2165-11-26 15:40:00,0.028333,98.400000000,108.000000000,16.000000000,98.000000000,160.000000000,70.000000000,0.0,2.000000000,"Chest pain, Wound eval"
5,10033085,38762319,23404293.0,2160-10-18 20:59:00,2160-10-19 12:03:49,ADMITTED,White,WALK IN,PACU,NaT,ED_WARD_DISCHARGE,M,61,2160,61,None,2160-10-19 12:02:00,2160-10-22 16:15:00,DIRECT EMER.,HOME HEALTH CARE,Private,English,MARRIED,NaN,2160-10-18 20:59:00,2160-10-22 16:15:00,0.030278,98.100000000,98.000000000,17.000000000,99.000000000,189.000000000,108.000000000,0.0,3.000000000,"Toe pain, Wound eval"
8,10040737,31711528,20352299.0,2155-08-10 21:12:00,2155-08-11 13:06:09,HOME,White,WALK IN,Surgery,NaT,ED_WARD_DISCHARGE,F,51,2149,57,None,2155-08-11 13:03:00,2155-08-16 13:47:00,DIRECT EMER.,HOME HEALTH CARE,Private,English,SINGLE,NaN,2155-08-10 21:12:00,2155-08-16 13:47:00,0.052500,97.300000000,72.000000000,18.000000000,97.000000000,115.000000000,71.000000000,4.0,3.000000000,"R Ankle injury, s/p Fall"
15,10160622,34447932,21082262.0,2180-04-20 17:08:00,2180-04-21 11:29:00,ADMITTED,White,AMBULANCE,Medical/Surgical Intensive Care Unit (MICU/SICU),2180-04-21 11:29:00,ED_DIRECT_ICU,F,64,2173,71,2181-03-27,2180-04-21 09:55:00,2180-05-13 18:14:00,DIRECT EMER.,SKILLED NURSING FACILITY,Medicare,English,SINGLE,NaN,2180-04-20 17:08:00,2180-05-13 18:14:00,1.566667,98.000000000,70.000000000,18.000000000,90.000000000,142.000000000,70.000000000,0.0,3.000000000,"Diarrhea, Weakness"
16,10160622,33586956,26936353.0,2180-06-30 21:25:00,2180-07-01 03:31:00,ADMITTED,White,AMBULANCE,Medical Intensive Care Unit (MICU),2180-07-01 03:31:00,ED_DIRECT_ICU,F,64,2173,71,2181-03-27,2180-07-01 02:17:00,2180-07-13 13:16:00,DIRECT EMER.,SKILLED NURSING FACILITY,Medicare,English,SINGLE,NaN,2180-06-30 21:25:00,2180-07-13 13:16:00,1.233333,99.300000000,72.000000000,18.000000000,96.000000000,162.000000000,58.000000000,0.0,3.000000000,Hypoxia


In [60]:
transfers = cohort_subset[['subject_id', 'ed_stay_id', 'hadm_id', "ed_intime", "ed_outtime", "first_icu_intime", "cohort_label", 'admittime', "dischtime", "stay_window_start", "stay_window_end", "admission_type"]]

In [61]:
transfers = transfers.copy()
for col in ['ed_intime', 'ed_outtime', 'first_icu_intime', 'admittime', 'dischtime', 'stay_window_start', 'stay_window_end']:
    transfers[col] = pd.to_datetime(transfers[col])

# For ICU patients, truncate stay_window_end to ICU admission (terminal state)
icu_mask = transfers['first_icu_intime'].notna()
transfers.loc[icu_mask, 'stay_window_end'] = transfers.loc[icu_mask, 'first_icu_intime']

# Terminal event: ICU transfer or discharge (ED or ward)
# Death is handled via reward penalty, not as a separate terminal state
def get_terminal_event(row):
    if pd.notna(row['first_icu_intime']):
        return 'ICU'
    elif pd.isna(row['admittime']):
        return 'DISCHARGE_ED'
    else:
        return 'DISCHARGE_WARD'  # dischtime == stay_window_end for these patients

transfers['terminal_event'] = transfers.apply(get_terminal_event, axis=1)

# Ward period: admittime â†’ stay_window_end (works for both ICU-truncated and ward discharge)
transfers['ward_start'] = transfers['admittime']  # NaN for ED-only patients
transfers['ward_end'] = transfers['stay_window_end']

print("Terminal event distribution:")
print(transfers['terminal_event'].value_counts())
print()
transfers[['ed_stay_id', 'subject_id', 'ed_intime', 'ed_outtime', 'ward_start', 'ward_end',
           'stay_window_end', 'terminal_event', 'admission_type']].head(10)

Terminal event distribution:
terminal_event
DISCHARGE_ED      38130
DISCHARGE_WARD    27834
ICU               18247
Name: count, dtype: int64



,ed_stay_id,subject_id,ed_intime,ed_outtime,ward_start,ward_end,stay_window_end,terminal_event,admission_type
0,34931809,10002013,2165-11-22 20:54:00,2165-11-23 08:20:42,2165-11-23 08:19:00,2165-11-23 08:20:42,2165-11-23 08:20:42,ICU,DIRECT EMER.
5,38762319,10033085,2160-10-18 20:59:00,2160-10-19 12:03:49,2160-10-19 12:02:00,2160-10-22 16:15:00,2160-10-22 16:15:00,DISCHARGE_WARD,DIRECT EMER.
8,31711528,10040737,2155-08-10 21:12:00,2155-08-11 13:06:09,2155-08-11 13:03:00,2155-08-16 13:47:00,2155-08-16 13:47:00,DISCHARGE_WARD,DIRECT EMER.
15,34447932,10160622,2180-04-20 17:08:00,2180-04-21 11:29:00,2180-04-21 09:55:00,2180-04-21 11:29:00,2180-04-21 11:29:00,ICU,DIRECT EMER.
16,33586956,10160622,2180-06-30 21:25:00,2180-07-01 03:31:00,2180-07-01 02:17:00,2180-07-01 03:31:00,2180-07-01 03:31:00,ICU,DIRECT EMER.
18,39172280,10189149,2165-06-26 17:58:00,2165-06-27 06:24:00,2165-06-27 05:17:00,2165-06-28 13:40:00,2165-06-28 13:40:00,DISCHARGE_WARD,DIRECT EMER.
20,32304446,10221321,2126-09-11 19:01:00,2126-09-11 23:18:00,2126-09-11 21:38:00,2126-09-11 23:18:00,2126-09-11 23:18:00,ICU,DIRECT EMER.
21,32871427,10232836,2126-02-14 16:33:00,2126-02-14 18:55:00,2126-02-14 14:00:00,2126-02-21 21:19:00,2126-02-21 21:19:00,DISCHARGE_WARD,DIRECT EMER.
22,30976465,10245573,2149-10-23 18:07:00,2149-10-24 01:18:00,2149-10-23 22:55:00,2149-10-28 15:15:00,2149-10-28 15:15:00,DISCHARGE_WARD,DIRECT EMER.
23,34640079,10250304,2144-08-03 06:29:00,2144-08-03 15:52:00,2144-08-03 13:04:00,2144-08-03 15:52:00,2144-08-03 15:52:00,ICU,DIRECT EMER.


In [89]:
transfers.reset_index(drop=True, inplace=True)

df_patient = df_patient.sort_values(['ed_stay_id', 'time']).reset_index(drop=True).reset_index().rename(columns={'index': 'step_idx'})

df_patient_sorted = df_patient[['ed_stay_id', 'subject_id', 'time', 'step_idx']]
stay_bounds = df_patient_sorted.groupby('ed_stay_id')['step_idx'].agg(first_idx='min', last_idx='max').reset_index()

# Snap transition times to nearest preceding time step
def snap_to_step(times_df, time_col, by_cols, step_col_name):
    return pd.merge_asof(
        times_df.dropna(subset=[time_col]).sort_values(time_col),
        df_patient_sorted.sort_values('time'),
        left_on=time_col,
        right_on='time',
        by=by_cols,
        direction='backward'
    ).rename(columns={'step_idx': step_col_name})

ed_end     = snap_to_step(transfers[['ed_stay_id', 'subject_id', 'ed_outtime']], 'ed_outtime', ['ed_stay_id', 'subject_id'], 'ed_end_step')
ward_start = snap_to_step(transfers[['ed_stay_id', 'subject_id', 'ward_start']].dropna(subset=['ward_start']), 'ward_start', ['ed_stay_id', 'subject_id'], 'ward_start_step')
ward_end   = snap_to_step(transfers[['ed_stay_id', 'subject_id', 'ward_end']].dropna(subset=['ward_end']), 'ward_end', ['ed_stay_id', 'subject_id'], 'ward_end_step')

# Combine all step indices
location_steps = (
    ed_end[['ed_stay_id', 'subject_id', 'ed_end_step']]
    .merge(ward_start[['ed_stay_id', 'subject_id', 'ward_start_step']], on=['ed_stay_id', 'subject_id'], how='left')
    .merge(ward_end[['ed_stay_id', 'subject_id', 'ward_end_step']], on=['ed_stay_id', 'subject_id'], how='left')
    .merge(stay_bounds, on='ed_stay_id', how='left')
    .merge(transfers[['ed_stay_id', 'terminal_event']].drop_duplicates('ed_stay_id'), on='ed_stay_id', how='left')
    .dropna(subset=['ed_end_step', 'first_idx', 'last_idx'])
)
location_steps[['ed_end_step', 'first_idx', 'last_idx']] = location_steps[['ed_end_step', 'first_idx', 'last_idx']].astype(int)

# If ward_start was before the patient's first time step, snap to first_idx
# Only for admitted patients (terminal_event != DISCHARGE_ED) to avoid catching ED-only patients
early_admit = (
    location_steps['ward_start_step'].isna() &
    location_steps['ward_end_step'].notna() &
    (location_steps['terminal_event'] != 'DISCHARGE_ED')
)
location_steps.loc[early_admit, 'ward_start_step'] = location_steps.loc[early_admit, 'first_idx']

del ed_end, ward_start, ward_end, stay_bounds
print('location_steps built')

location_steps built


In [91]:
# Helper: vectorized range expansion
def expand_ranges(df, start_col, end_col, flag_col):
    starts = df[start_col]
    ends = df[end_col]
    lengths = (ends - starts + 1).clip(lower=0)
    if lengths.sum() == 0:
        return pd.DataFrame({'step_idx': pd.Series(dtype=int), flag_col: pd.Series(dtype=int)})
    offsets = np.arange(lengths.sum())
    offsets -= np.repeat(lengths.cumsum().shift(1, fill_value=0).values, lengths.values)
    return pd.DataFrame({
        'step_idx': (np.repeat(starts.values, lengths.values) + offsets).astype(int),
        flag_col: 1
    })

admitted = location_steps.dropna(subset=['ward_start_step']).copy()
admitted['ward_start_step'] = admitted['ward_start_step'].astype(int)
admitted['ed_end'] = admitted['ward_start_step'] - 1

ed_only = location_steps[location_steps['ward_start_step'].isna()].copy()
ed_only['ed_end'] = ed_only['last_idx'].astype(int)

# in_ed: build, assign, free
ed_df = pd.concat([
    expand_ranges(admitted.assign(ed_start=admitted['first_idx']), 'ed_start', 'ed_end', 'in_ed'),
    expand_ranges(ed_only.assign(ed_start=ed_only['first_idx']), 'ed_start', 'ed_end', 'in_ed')
])
df_patient['in_ed'] = df_patient['step_idx'].isin(ed_df['step_idx']).astype(int)
del ed_df, ed_only
print('in_ed done')

# in_ward: build, assign, free
ward_rows = admitted.copy()
ward_rows['last_idx'] = ward_rows['last_idx'].astype(int)
ward_df = expand_ranges(ward_rows, 'ward_start_step', 'last_idx', 'in_ward')
df_patient['in_ward'] = df_patient['step_idx'].isin(ward_df['step_idx']).astype(int)
del ward_df
print('in_ward done')

# ed_boarding: build, assign, free
boarding_rows = admitted[admitted['ward_start_step'] <= admitted['ed_end_step']].copy()
boarding_df = expand_ranges(boarding_rows, 'ward_start_step', 'ed_end_step', 'ed_boarding')
df_patient['ed_boarding'] = df_patient['step_idx'].isin(boarding_df['step_idx']).astype(int)
del boarding_df, admitted, boarding_rows, ward_rows
print('ed_boarding done')

# Override ED-only patients (no hadm_id) — force in_ed=1, in_ward=0, ed_boarding=0
ed_only_stays = set(transfers[transfers['terminal_event'] == 'DISCHARGE_ED']['ed_stay_id'])
ed_only_mask = df_patient['ed_stay_id'].isin(ed_only_stays)
df_patient.loc[ed_only_mask, 'in_ed'] = 1
df_patient.loc[ed_only_mask, 'in_ward'] = 0
df_patient.loc[ed_only_mask, 'ed_boarding'] = 0

# Add terminal_event and drop step_idx
df_patient = df_patient.merge(
    transfers[['ed_stay_id', 'terminal_event']].drop_duplicates('ed_stay_id'),
    on='ed_stay_id',
    how='left'
).drop(columns='step_idx')

del location_steps, df_patient_sorted
df_patient = df_patient.copy()  # defragment
print('Done')

in_ed done
in_ward done
ed_boarding done
Done


In [ ]:
# Checks
neither = df_patient[(df_patient['in_ed'] == 0) & (df_patient['in_ward'] == 0)]
print(f'Rows with neither in_ed nor in_ward: {len(neither)}')  # should be 0
del neither
both = df_patient[(df_patient['in_ed'] == 1) & (df_patient['in_ward'] == 1)]
print(f'Rows with both in_ed and in_ward:    {len(both)}')     # should be 0
del both

Rows with neither in_ed nor in_ward: 0
Rows with both in_ed and in_ward:    0
Duplicate (stay, time) rows:         21684


In [93]:
# Merge stay_window_end (already truncated to first_icu_intime for ICU patients)
df_patient = df_patient.merge(
    transfers[['ed_stay_id', 'stay_window_end']],
    on='ed_stay_id',
    how='left'
)

df_patient['time'] = pd.to_datetime(df_patient['time'])
df_patient['stay_window_end'] = pd.to_datetime(df_patient['stay_window_end'])

# Drop rows past stay_window_end
df_patient = df_patient[df_patient['time'] <= df_patient['stay_window_end']].copy().reset_index(drop=True)

# For ICU patients, update the last row's time to exactly first_icu_intime
icu_stay_ids = set(transfers[transfers['terminal_event'] == 'ICU']['ed_stay_id'])
last_row_mask = ~df_patient.duplicated(subset='ed_stay_id', keep='last')
icu_last_mask = last_row_mask & df_patient['ed_stay_id'].isin(icu_stay_ids)
df_patient.loc[icu_last_mask, 'time'] = df_patient.loc[icu_last_mask, 'stay_window_end']

df_patient.drop(columns='stay_window_end', inplace=True)

# Verify
sample_icu = transfers[transfers['terminal_event'] == 'ICU']['ed_stay_id'].iloc[0]
n_rows = len(df_patient[df_patient['ed_stay_id'] == sample_icu])
icu_end = transfers[transfers['ed_stay_id'] == sample_icu]['stay_window_end'].iloc[0]
icu_start = transfers[transfers['ed_stay_id'] == sample_icu]['stay_window_start'].iloc[0]
print(f"Stay: {sample_icu}, rows: {n_rows}, start: {icu_start}, terminal: {icu_end}")
print(f"Last time step: {df_patient[df_patient['ed_stay_id'] == sample_icu]['time'].iloc[-1]}")

Stay: 34931809, rows: 23, start: 2165-11-22 20:54:00, terminal: 2165-11-23 08:20:42
Last time step: 2165-11-23 08:20:42


In [94]:
dups = df_patient.duplicated(subset=['ed_stay_id', 'subject_id', 'hadm_id', 'time']).sum()
print(f'Duplicate (stay, time) rows:         {dups}')          # should be 0
del dups

Duplicate (stay, time) rows:         20366


# Merge Everything Else

In [95]:
height = mimic_loader(path=root, name='height')
df_medrecon = load_dataset(root_2, "combined_medication_features", split='med_only').to_pandas()

heights = height.groupby('subject_id').agg(
    height=("result_value", "mean")
).round(2).reset_index()

del height

In [96]:
cohort_merge_columns = ['ed_stay_id', 'gender', 'anchor_age', 'arrival_transport', 'acuity']

medrecon_merge_columns = ['ed_stay_id'] + [c for c in df_medrecon.columns if c.startswith('recon')][:-2]

# Merge cohort static features
df_patient = df_patient.merge(df_cohort[cohort_merge_columns], how='left', on='ed_stay_id')

# Merge height (subject-level, doesn't change across stays)
df_patient = df_patient.merge(heights, how='left', on='subject_id')

# Merge medication reconciliation (pre-ED medications, static per stay)
df_patient = df_patient.merge(df_medrecon[medrecon_merge_columns], how='left', on='ed_stay_id')

# admission_type: only populated when patient is in ward, NaN otherwise
df_patient = df_patient.merge(transfers[['ed_stay_id', "admission_type"]], on='ed_stay_id', how='left')
df_patient.loc[df_patient['in_ward'] == 0, 'admission_type'] = np.nan


In [97]:
del df_medrecon

# NA Check/Event Overlap check

In [271]:
df_patient.to_parquet("data/full_df_patient.parquet", index=False)

In [ ]:
from datasets import Dataset
from utils.hub_card import push_dataset_card

DESCRIPTION = (
    "Full patient state dataframe for offline RL training, derived from MIMIC-IV ED data. "
    "One row per 30-minute time block per ED stay (ed_stay_id). "
    "Built by combining: lab order/result statuses (0=not ordered, 1=pending, 2=normal, 3=abnormal), "
    "microbiology order/result statuses (0-4), vital signs with fill-forward and delta/rate features, "
    "medication administration flags (binary, first admin onward), ECG status, weight, "
    "and transfer location flags (in_ed, in_ward, ed_boarding). "
    "Static features merged per stay: gender, age, arrival transport, triage acuity, admission type (ward rows only), "
    "height, and pre-ED medication reconciliation (recon_* drug classes). "
    "terminal_event column encodes the outcome: ICU transfer, ward discharge, or ED discharge. "
    "Intended as the primary input for offline RL state construction."
)

ds = Dataset.from_pandas(df_patient, split='full_patient_state')
ds.push_to_hub("ADS599-Capstone/modeling_data", config_name="full_patient_state", data_dir="full_patient_state")
push_dataset_card("ADS599-Capstone/modeling_data", config_name="full_patient_state", split="full_patient_state", data_dir="full_patient_state", description=DESCRIPTION)
print("Pushed full patient state to HuggingFace Hub.")

Many of the rate calculations are NaN as they should be (maybe) for first row in stay but some are NaN beyond that.  Could recalculate

what to do about overlapping actions?

In [ ]:
# About 30k na's for this one
df_patient['time_since_last_hrs'] = df_patient['time_since_last_hrs'].fillna(0)

In [101]:
# About 20k duplicates from somewhere, probably when merged with transfers
print(df_patient.duplicated(subset=['ed_stay_id', "subject_id", "hadm_id", "time"]).sum())
df_patient.drop_duplicates(subset=['ed_stay_id', "subject_id", "hadm_id", "time"], inplace=True)

20366


# Observe

In [ ]:
df_patient.reset_index(drop=True, inplace=True)

(6551723, 129)


In [ ]:
med_cols = [
    'Other', 'Analgesic - Opioid/NSAID', 'Antiemetic', 'Analgesic - Acetaminophen',
    'Antibiotic', 'Benzodiazepine - Sedative/Anxiolytic', 'Analgesic - NSAID',
    'Bronchodilator', 'Antiplatelet', 'GI - Acid Suppression', 'Corticosteroid',
    'Beta Blocker', 'Diuretic', 'Antipsychotic', 'Anticoagulant', 'Insulin/Glucose',
    'Calcium Channel Blocker', 'Nitrate', 'ACE Inhibitor', 'IV Fluid',
    'Anticonvulsant', 'Antiarrhythmic'
]

# Detect 0->positive transitions within each stay for ecg, rad, in_ward
grp = df_patient.groupby('ed_stay_id')
ecg_prev  = grp['ecg_status'].shift(1, fill_value=0)
rad_prev  = grp['rad_status'].shift(1, fill_value=0)
ward_prev = grp['in_ward'].shift(1, fill_value=0)

ecg_ordered  = (df_patient['ecg_status'] > 0) & (ecg_prev == 0)
rad_ordered  = (df_patient['rad_status'] > 0) & (rad_prev == 0)
ward_transition = (df_patient['in_ward'] == 1) & (ward_prev == 0)

# Add the ecg and rad action columns
df_patient['ecg_ordered'] = ecg_ordered.astype(int)
df_patient['rad_ordered'] = rad_ordered.astype(int)

# Meds: 0->1 transition within stay (first administration)
med_prev = grp[med_cols].shift(1).fillna(0)
med_ordered = (df_patient[med_cols].values > med_prev.values).any(axis=1)

# Terminal row per stay always has an action (discharge or ICU transfer)
last_row = ~df_patient.duplicated(subset='ed_stay_id', keep='last')

# OBSERVE = 1 when no action was taken this step
any_action = (
    (df_patient['labs_ordered'] == 1) |
    (df_patient['micro_ordered'] == 1) |
    (df_patient['vitals_checked'] == 1) |
    ecg_ordered | rad_ordered | ward_transition | med_ordered | last_row
)
df_patient['observe'] = (~any_action).astype(int)

print(f"Observe rate: {df_patient['observe'].mean():.2%}")
print(f"Action breakdown:")
print(f"  labs_ordered:    {df_patient['labs_ordered'].sum():>10,}")
print(f"  micro_ordered:   {df_patient['micro_ordered'].sum():>10,}")
print(f"  vitals_checked:  {df_patient['vitals_checked'].sum():>10,}")
print(f"  ecg_ordered:     {ecg_ordered.sum():>10,}")
print(f"  rad_ordered:     {rad_ordered.sum():>10,}")
print(f"  ward_transition: {ward_transition.sum():>10,}")
print(f"  med_ordered:     {med_ordered.sum():>10,}")
print(f"  terminal:        {last_row.sum():>10,}")
print(f"  observe:         {df_patient['observe'].sum():>10,}")


Observe rate: 87.57%
Action breakdown:
  labs_ordered:       313,144
  micro_ordered:       36,072
  vitals_checked:   344,167.0
  ecg_ordered:         32,358
  rad_ordered:         53,702
  ward_transition:     46,079
  med_ordered:        101,874
  terminal:            84,211
  observe:          5,737,095


In [179]:
# Mismatch after transfers where 84209 out of 84211 ed stays were coded in_ward even though it should be half that.  during transfer merge, ED_DISCHARGE patients were all being labeled as in ward
# because of the code to indicate patients that were boarded in the ED.  This is the temp fix

# Override ED-only patients (no hadm_id) — force in_ed=1, in_ward=0, ed_boarding=0
ed_only_stays = set(transfers[transfers['terminal_event'] == 'DISCHARGE_ED']['ed_stay_id'])
ed_only_mask = df_patient['ed_stay_id'].isin(ed_only_stays)
df_patient.loc[ed_only_mask, 'in_ed'] = 1
df_patient.loc[ed_only_mask, 'in_ward'] = 0
df_patient.loc[ed_only_mask, 'ed_boarding'] = 0

# BCQ Test Set

In [ ]:
# This reuses some of the some logic as above so we can condense that

med_cols = [
    'Other', 'Analgesic - Opioid/NSAID', 'Antiemetic', 'Analgesic - Acetaminophen',
    'Antibiotic', 'Benzodiazepine - Sedative/Anxiolytic', 'Analgesic - NSAID',
    'Bronchodilator', 'Antiplatelet', 'GI - Acid Suppression', 'Corticosteroid',
    'Beta Blocker', 'Diuretic', 'Antipsychotic', 'Anticoagulant', 'Insulin/Glucose',
    'Calcium Channel Blocker', 'Nitrate', 'ACE Inhibitor', 'IV Fluid',
    'Anticonvulsant', 'Antiarrhythmic'
]

grp = df_patient.groupby('ed_stay_id')

# dispense_meds: any med column transitioned 0->1 this step
med_prev = grp[med_cols].shift(1).fillna(0)
df_patient['dispense_meds'] = (df_patient[med_cols].values > med_prev.values).any(axis=1).astype(int)

# ward_transfer: in_ward 0->1 transition within stay
ward_prev = grp['in_ward'].shift(1, fill_value=0)
df_patient['ward_transfer'] = ((df_patient['in_ward'] == 1) & (ward_prev == 0)).astype(int)

# discharge / transfer_icu: terminal row only
last_row = ~df_patient.duplicated(subset='ed_stay_id', keep='last')
df_patient['discharge'] = (last_row & df_patient['terminal_event'].isin(['DISCHARGE_ED', 'DISCHARGE_WARD'])).astype(int)
df_patient['transfer_icu'] = (last_row & (df_patient['terminal_event'] == 'ICU')).astype(int)

action_cols = ['observe', 'vitals_checked', 'labs_ordered', 'micro_ordered',
               'ecg_ordered', 'rad_ordered', 'dispense_meds',
               'ward_transfer', 'discharge', 'transfer_icu']

print("Action distribution:")
for col in action_cols:
    if col in df_patient.columns:
        print(f"  {col:<20} {df_patient[col].sum():>10,}  ({df_patient[col].mean():.2%})")

# Multi-label check — exclude observe since it's mutually exclusive with all actions
action_only_cols = [c for c in action_cols if c != 'observe']
action_counts = df_patient[action_only_cols].sum(axis=1)
print("Actions per time step (excluding observe):")
print(action_counts.value_counts().sort_index().to_string())
print(f"Rows with 2+ simultaneous actions: {(action_counts >= 2).sum():,}  ({(action_counts >= 2).mean():.2%})")


Action distribution:
  observe               5,737,095  (87.57%)
  vitals_checked        344,167.0  (5.25%)
  labs_ordered            313,144  (4.78%)
  micro_ordered            36,072  (0.55%)
  dispense_meds           101,874  (1.55%)
  ward_transfer            46,079  (0.70%)
  discharge                65,964  (1.01%)
  transfer_icu             18,247  (0.28%)


In [193]:
for col in action_cols:
    if col in df_patient.columns:
        print(f"  {col:<20} {df_patient[col].sum():>10,}  ({df_patient[col].mean():.2%})")

# Multi-label check — exclude observe since it's mutually exclusive with all actions
action_only_cols = [c for c in action_cols if c != 'observe']
action_counts = df_patient[action_only_cols].sum(axis=1)
print("\nActions per time step (excluding observe):")
print(action_counts.value_counts().sort_index().to_string())
print(f"\nRows with 2+ simultaneous actions: {(action_counts >= 2).sum():,}  ({(action_counts >= 2).mean():.2%})")

  observe               5,737,095  (87.57%)
  vitals_checked        344,167.0  (5.25%)
  labs_ordered            313,144  (4.78%)
  micro_ordered            36,072  (0.55%)
  ecg_ordered              32,358  (0.49%)
  rad_ordered              53,702  (0.82%)
  dispense_meds           101,874  (1.55%)
  ward_transfer            46,079  (0.70%)
  discharge                65,964  (1.01%)
  transfer_icu             18,247  (0.28%)

Actions per time step (excluding observe):
0.0    5737095
1.0     655739
2.0     126766
3.0      26894
4.0       4550
5.0        625
6.0         49
7.0          5

Rows with 2+ simultaneous actions: 158,889  (2.43%)


In [201]:
df_patient.head()

,ed_stay_id,subject_id,hadm_id,time,Chemistry-Blood,Hematology-Blood,Hematology-Urine,Blood Gas-Blood,Chemistry-Urine,Chemistry-Other Body Fluid,Hematology-Cerebrospinal Fluid,Chemistry-Cerebrospinal Fluid,Hematology-Ascites,Chemistry-Ascites,Chemistry-Pleural,Hematology-Pleural,Hematology-Joint Fluid,Chemistry-Stool,Hematology-Other Body Fluid,Blood Gas-Other Body Fluid,Hematology-Bone Marrow,Hematology-Stool,Chemistry-Joint Fluid,labs_ordered,URINE,Rapid Respiratory Viral Screen & Culture,PERITONEAL FLUID,BLOOD CULTURE,STOOL,OTHER,PLEURAL FLUID,TISSUE,SEROLOGY/BLOOD,IMMUNOLOGY,MRSA SCREEN,SWAB,JOINT FLUID,SPUTUM,"FLUID,OTHER",ABSCESS,BRONCHOALVEOLAR LAVAGE,CSF;SPINAL FLUID,Blood (CMV AB),Blood (EBV),BILE,micro_ordered,Other,Analgesic - Opioid/NSAID,Antiemetic,Analgesic - Acetaminophen,Antibiotic,Benzodiazepine - Sedative/Anxiolytic,Analgesic - NSAID,Bronchodilator,Antiplatelet,GI - Acid Suppression,Corticosteroid,Beta Blocker,Diuretic,Antipsychotic,Anticoagulant,Insulin/Glucose,Calcium Channel Blocker,Nitrate,ACE Inhibitor,IV Fluid,Anticonvulsant,Antiarrhythmic,weight,ecg_status,rad_status,in_ed,in_ward,ed_boarding,terminal_event,gender,anchor_age,arrival_transport,acuity,height,recon_ace_inhibitor,recon_analgesic___nsaid,recon_analgesic___opioid_nsaid,recon_antiarrhythmic,recon_antibiotic,recon_anticoagulant,recon_anticonvulsant,recon_antiemetic,recon_antiplatelet,recon_antipsychotic,recon_benzodiazepine___sedative_anxiolytic,recon_beta_blocker,recon_bronchodilator,recon_calcium_channel_blocker,recon_corticosteroid,recon_diuretic,recon_gi___acid_suppression,recon_insulin_glucose,recon_nitrate,admission_type,vitals_checked,current_temperature,current_heartrate,current_resprate,current_o2sat,current_sbp,current_dbp,current_pain,current_pulse_pressure,current_map,time_since_last_hrs,temperature_rolling2h,temperature_delta,temperature_rate,heartrate_rolling2h,heartrate_delta,heartrate_rate,resprate_rolling2h,resprate_delta,resprate_rate,o2sat_rolling2h,o2sat_delta,o2sat_rate,sbp_rolling2h,sbp_delta,sbp_rate,dbp_rolling2h,dbp_delta,dbp_rate,observe,dispense_meds,ward_transfer,discharge,transfer_icu,ecg_ordered,rad_ordered
0,30000012,11714491,21562392.0,2126-02-14 20:22:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,1,0,0,DISCHARGE_WARD,F,65,AMBULANCE,2.000000000,65.0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,1,0,0,NaN,1.0,98.8,96.0,18.0,93.0,160.0,54.0,0.0,106.0,89.33,0.0,98.8,0.0,NaN,96.0,0.0,NaN,18.0,0.0,NaN,93.0,0.0,NaN,160.0,0.0,NaN,54.0,0.0,NaN,0,0,0,0,0,0,0
1,30000012,11714491,21562392.0,2126-02-14 20:52:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,3.0,0.0,1,0,0,DISCHARGE_WARD,F,65,AMBULANCE,2.000000000,65.0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,1,0,0,NaN,0.0,98.8,96.0,18.0,93.0,160.0,54.0,0.0,106.0,89.33,0.5,98.8,0.0,NaN,96.0,0.0,NaN,18.0,0.0,NaN,93.0,0.0,NaN,160.0,0.0,NaN,54.0,0.0,NaN,0,0,0,0,0,1,0
2,30000012,11714491,21562392.0,2126-02-14 21:22:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,3.0,0.0,1,0,0,DISCHARGE_WARD,F,65,AMBULANCE,2.000000000,65.0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,1,0,0,NaN,0.0,98.8,96.0,18.0,93.0,160.0,54.0,0.0,106.0,89.33,1.0,98.8,0.0,NaN,96.0,0.0,NaN,18.0,0.0,NaN,93.0,0.0,NaN,160.0,0.0,NaN,54.0,0.0,NaN,1,0,0,0,0,0,0
3,30000012,11714491,21562392.0,2126-02-14 21:52:00,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0

In [ ]:
# Cluster all the action columns at end

move_to_end = ['vitals_checked', 'labs_ordered', 'micro_ordered']
df_patient = df_patient[[c for c in df_patient.columns if c not in move_to_end] + move_to_end]